# MonsoonCast: Notebook 01 — Exploratory Data Analysis & Monsoon Climatology

**Author:** Khushaal Srivastava  
**Project:** MonsoonCast  
**Date:** February 23, 2026  

This notebook inspects IMD 0.25° gridded daily rainfall data, explores summer monsoon (JJAS) seasonal dynamics over Central India, evaluates extreme event class imbalance, and checks spatial orographic gradients.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns

# Add src to Python path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from monsooncast.data.loaders import generate_synthetic_monsoon_dataset, standardize_dataset_coords
from monsooncast.data.validation import validate_coordinates, audit_missing_values

## 1. Load and Inspect Meteorological Array

In [ ]:
data_path = Path("../data/raw/synthetic_sample.nc")
if not data_path.exists():
    print("Generating test fixture dataset...")
    ds = generate_synthetic_monsoon_dataset(output_path=data_path)
else:
    ds = xr.open_dataset(data_path)

ds = standardize_dataset_coords(ds)
print("Dimensions:", ds.sizes)
print("Coordinates:", list(ds.coords.keys()))
print("Data Variables:", list(ds.data_vars.keys()))

## 2. Spatial Climatology & Orographic Enhancement

In [ ]:
# Mean daily monsoon rainfall across spatial grid
mean_rain = ds["rain"].mean(dim="time")

plt.figure(figsize=(8, 6))
mean_rain.plot.contourf(cmap="YlGnBu", levels=15)
plt.title("Mean Daily Monsoon Rainfall (JJAS) [mm/day]")
plt.xlabel("Longitude (°E)")
plt.ylabel("Latitude (°N)")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

## 3. Rainfall Distribution and Extreme Tail

In [ ]:
rain_flat = ds["rain"].values.flatten()
wet_rain = rain_flat[wet_mask := (rain_flat > 0.1)]

print(f"Total Grid-Days: {len(rain_flat):,}")
print(f"Wet Days (>0.1 mm): {len(wet_rain):,} ({len(wet_rain)/len(rain_flat)*100:.1f}%)")
print(f"Heavy Rain (>=50 mm): {np.sum(rain_flat >= 50):,} ({np.mean(rain_flat >= 50)*100:.2f}%)")
print(f"Very Heavy Rain (>=100 mm): {np.sum(rain_flat >= 100):,} ({np.mean(rain_flat >= 100)*100:.2f}%)")

plt.figure(figsize=(9, 4))
plt.hist(wet_rain, bins=60, density=True, color="royalblue", alpha=0.7, log=True)
plt.axvline(50.0, color="red", linestyle="--", label="IMD Heavy Rain Threshold (50 mm/day)")
plt.title("Wet-Day Daily Rainfall Distribution (Log-Scale Density)")
plt.xlabel("Precipitation (mm/day)")
plt.ylabel("Log Probability Density")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()